# Stage A Market Research Agent - Colab Runner

Notebook này chạy dự án MarketMind-AI trên Google Colab.
Workflow:
1. Clone GitHub repo
2. Kết nối Google Drive
3. Load environment variables từ .env
4. Chạy pipeline từ các tệp .py
5. Mở Flask API server với Ngrok tunnel

## 1. Clone Repository from GitHub

In [ ]:
import os
import sys

# Clone repo
!git clone https://github.com/vien10022003/MarketMind-AI.git

# Navigate to backend directory
os.chdir('/content/MarketMind-AI/backend')
sys.path.insert(0, '/content/MarketMind-AI/backend')

print(f"Current directory: {os.getcwd()}")
print("Repository cloned successfully!")

## 2. Mount Google Drive and Load Environment Variables

In [ ]:
from google.colab import drive

# Mount Google Drive
drive.mount('/content/drive')

# Load environment variables from .env file
from dotenv import load_dotenv

env_path = "/content/drive/MyDrive/Colab Notebooks/AI/do an/.env"
if os.path.exists(env_path):
    load_dotenv(env_path)
    print(f"✅ Environment variables loaded from {env_path}")
else:
    print(f"❌ .env file not found at {env_path}")
    print("Please check the path and try again.")

# Verify required keys are loaded
required_keys = ["LANGCHAIN_API_KEY", "TAVILY_API_KEY"]
missing = [k for k in required_keys if not os.getenv(k)]
if missing:
    print(f"⚠️  Missing keys: {missing}")
else:
    print(f"✅ All required environment variables loaded")

## 3. Install Dependencies and Verify Setup

In [ ]:
# Step 1: Downgrade pandas về version Colab compatible
!pip install --no-deps pandas==2.2.2 -q

# Step 2: Install pydantic with correct dependencies
!pip install pydantic==2.10.3 -q

# Step 3: Fix typing_extensions
!pip install "typing-extensions>=4.12.2" --upgrade -q

# Step 4: Install LangChain core dependencies
!pip install langchain-core==0.3.24 langchain-text-splitters==0.3.2 langsmith==0.1.82 --no-deps -q

# Step 5: Install LangChain + dependencies
!pip install langchain==0.3.11 langchain-community==0.3.5 langchain-tavily==0.2.14 tavily-python==0.7.23 langgraph==0.2.17 python-dotenv==1.0.1 rich==13.8.1 --no-deps -q

# Step 6: Install Flask + Ngrok
!pip install pyngrok==7.5.1 flask-cors==6.0.2 pymongo flask --no-deps -q
!pip install pymongo flask -q

# Step 7: Verify imports
print("Testing imports...")
import pandas as pd
import torch
from pydantic import BaseModel, Field
from rich import print as rprint
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline
from langchain_tavily import TavilySearch
from flask import Flask
from pyngrok import ngrok
import pymongo

rprint(f"[green]✅ Pydantic version: {__import__('pydantic').__version__}")
rprint(f"[green]✅ LangChain version: {__import__('langchain').__version__}")
rprint(f"[green]✅ All dependencies installed successfully![/green]")

## 4. Initialize Local LLM and Tavily Tool

In [ ]:
from stage_a.llm_config import LocalLLMConfig, LocalTextGenerator
from stage_a.environment import load_environment

# Setup environment and logging
load_environment()

# Initialize LLM
llm_cfg = LocalLLMConfig()
print(f"Initializing Local LLM: {llm_cfg.model_name}")
local_llm = LocalTextGenerator(llm_cfg)

rprint("[green]✅ Local LLM initialized successfully[/green]")

# Initialize Tavily
from stage_a.tavily_search import initialize_tavily
tavily_tool = initialize_tavily()
rprint("[green]✅ Tavily search tool initialized[/green]")

## 5. Define Data Models and Configuration

In [ ]:
from typing import List, Optional, Dict, Any
from stage_a.data_models import StageAInput, EvidenceItem, StageAOutput

rprint("[green]✅ Data models imported successfully[/green]")

# Test model instantiation
test_input = StageAInput(
    user_prompt="Test query",
    nganh_hang="Test Industry",
    thi_truong_muc_tieu="Test Market"
)
print(f"Test input created: {test_input.user_prompt}")

## 6. Load LLM Clarification and Validation Functions

In [ ]:
from stage_a.clarification import (
    validate_input_completeness,
    request_additional_information,
    clarify_user_prompt
)

rprint("[green]✅ Clarification functions loaded[/green]")

## 7. Load Planning Chain Function

In [ ]:
from stage_a.planning import planner_chain

rprint("[green]✅ Planning chain function loaded[/green]")

## 8. Load ReAct Agent Loop Function

In [ ]:
from stage_a.react import (
    react_decide_action,
    run_react_loop
)

rprint("[green]✅ ReAct agent functions loaded[/green]")

## 9. Load Evidence Normalization and Filtering

In [ ]:
from stage_a.evidence_processing import (
    normalize_and_filter_evidence,
    score_source
)

rprint("[green]✅ Evidence processing functions loaded[/green]")

## 10. Load Report Synthesis Functions

In [ ]:
from stage_a.synthesis import (
    synthesize_tong_quan_thi_truong,
    synthesize_phan_tich_doi_thu,
    synthesize_xu_huong_nganh,
    synthesize_phan_khuc_va_insight,
    synthesize_stage_a_report
)
from stage_a.output_formatting import (
    build_markdown_report,
    convert_evidence_to_dict
)

rprint("[green]✅ Report synthesis functions loaded[/green]")

## 11. Configure MongoDB Connection

In [ ]:
from stage_a.mongodb import MongoDBManager

# Initialize MongoDB client
mongo_uri = os.getenv("MONGO_URI")
if mongo_uri:
    mongo_manager = MongoDBManager(uri=mongo_uri)
    rprint("[green]✅ MongoDB connection initialized[/green]")
else:
    mongo_manager = None
    rprint("[yellow]⚠️  MONGO_URI not set - MongoDB persistence disabled[/yellow]")

## 12. Initialize Flask API Server with Streaming

In [ ]:
from stage_a.flask_api import app

rprint("[green]✅ Flask API server initialized[/green]")
rprint("[bold]Endpoints available:[/bold]")
rprint("  - POST /api/research/stage_a  (Research execution with streaming)")
rprint("  - GET  /health               (Server health check)")

## 13. Start Ngrok Tunnel and Run Server

In [ ]:
from stage_a.ngrok_tunnel import setup_ngrok_tunnel
from werkzeug.serving import WSGIRequestHandler

# Setup Ngrok
ngrok_auth_token = os.getenv("NGROK_AUTH_TOKEN")
if not ngrok_auth_token:
    rprint("[red]❌ NGROK_AUTH_TOKEN not found in .env[/red]")
    rprint("    Please add NGROK_AUTH_TOKEN to your .env file")
else:
    try:
        # Set ngrok auth token
        from pyngrok import ngrok as ngrok_module
        ngrok_module.set_auth_token(ngrok_auth_token)
        
        # Increase timeout
        WSGIRequestHandler.timeout = 3600
        
        # Create tunnel
        public_url = ngrok_module.connect(5000, bind_tls=True)
        
        rprint(f"[bold green]✅ Ngrok tunnel created![/bold green]")
        rprint(f"[bold green]Public URL: {public_url}[/bold green]")
        rprint(f"[bold green]API Endpoint: {public_url}/api/research/stage_a[/bold green]")
        rprint(f"[bold green]Server timeout: 3600 seconds (1 hour)[/bold green]")
        
        # Start Flask server
        print("\n" + "="*60)
        print("Starting Flask API Server...")
        print("="*60)
        print(f"\n🚀 Server running at: http://0.0.0.0:5000")
        print(f"🌐 Public URL: {str(public_url)}")
        print(f"📊 API Endpoint: {str(public_url)}/api/research/stage_a")
        print(f"❤️  Health Check: {str(public_url)}/health\n")
        
        # Run the server
        app.run(host='0.0.0.0', port=5000, debug=False, use_reloader=False, threaded=True)
        
    except Exception as e:
        rprint(f"[red]❌ Error starting server: {str(e)}[/red]")
        import traceback
        traceback.print_exc()

## Optional: Test Pipeline Locally (Without API)

Run this cell to test the pipeline locally without starting the API server.

In [ ]:
# Optional: Test the pipeline with a sample query
# Uncomment below to run

# from stage_a.main import run_pipeline
# 
# # Example research query
# output = run_pipeline(
#     user_prompt="Tìm hiểu thị trường xe điện tại Việt Nam",
#     industry="Xe điện",
#     market="Việt Nam"
# )
# 
# print("\n" + "="*60)
# print("STAGE A RESEARCH REPORT")
# print("="*60)
# print(output.tong_quan_thi_truong[:500])
# print(f"\nCitations: {len(output.citations)} sources collected")